# Input formats

ScisTreeCNA takes a numeric array of shape `(n_sites, n_cells, 3)` or
`(n_sites, n_cells, 4)`. This notebook shows the three ways to produce one — a CSV of
read counts, a VCF, or building the array directly — and how missing data is encoded.

In [1]:
import numpy as np
import pandas as pd
import scistreecna as scna

## CSV with total copy number

The usual input. Rows are sites, columns are cells, the first column is the site index,
and the header row holds the cell names. Every entry is a `|`-delimited string:

```text
ref|alt|cn
```

| Field | Meaning |
| :--- | :--- |
| `ref` | read count supporting the reference (wild-type) allele |
| `alt` | read count supporting the mutant allele |
| `cn` | observed copy number — absolute copy number is recommended, but a relative copy state also works |

:::{important}
Rows are **sites** and columns are **cells**. Getting this the wrong way round produces a
tree over your sites.
:::

In [2]:
raw = pd.read_csv('data/test_data_reads.csv', index_col=0)
print("table shape (sites x cells):", raw.shape)
raw.iloc[:4, :5]

table shape (sites x cells): (100, 60)


,c0,c1,c2,c3,c4
s0,11|0|2,20|1|1,25|0|2,9|0|2,36|0|2
s1,4|0|2,23|0|1,18|0|2,14|2|2,32|0|2
s2,13|0|2,26|0|2,20|0|2,6|0|2,17|0|2
s3,15|1|3,12|0|3,15|0|2,3|0|2,20|0|3


In [3]:
reads, cell_names, site_names = scna.util.read_csv('data/test_data_reads.csv')

print("parsed array shape:", reads.shape)      # (n_sites, n_cells, 3)
print("dtype:", reads.dtype)
print()
print("raw string at (s0, c0):", raw.iloc[0, 0])
print("parsed  [ref alt cn] :", reads[0, 0])

parsed array shape: (100, 60, 3)
dtype: object

raw string at (s0, c0): 11|0|2
parsed  [ref alt cn] : [11 0 2]


## Missing values

A `.` marks a missing field. Reads and copy number can go missing independently:

| Entry | Meaning |
| :--- | :--- |
| `.\|.\|cn` | read counts missing, copy number observed |
| `ref\|alt\|.` | read counts observed, copy number missing |
| `.\|.\|.` | both missing |

On parsing, missing **read counts become `0`** and a missing **copy number becomes `-1`**.
The `-1` sentinel is what tells the model to ignore the copy-number term for that entry
rather than to believe a copy number of zero.

In [4]:
toy = pd.DataFrame(
    [['12|0|2', '.|.|2',  '8|3|.'],
     ['5|5|3',  '.|.|.',  '0|9|2']],
    index=['s0', 's1'],
    columns=['c0', 'c1', 'c2'],
)
toy.to_csv('missing_demo.csv')
toy

,c0,c1,c2
s0,12|0|2,.|.|2,8|3|.
s1,5|5|3,.|.|.,0|9|2


In [5]:
toy_reads, toy_cells, toy_sites = scna.util.read_csv('missing_demo.csv')

for i, s in enumerate(toy_sites):
    for j, c in enumerate(toy_cells):
        print(f"{s},{c}  {toy.iloc[i, j]:>8}  ->  {toy_reads[i, j]}")

s0,c0    12|0|2  ->  [12 0 2]
s0,c1     .|.|2  ->  [0 0 2]
s0,c2     8|3|.  ->  [8 3 -1]
s1,c0     5|5|3  ->  [5 5 3]
s1,c1     .|.|.  ->  [0 0 -1]
s1,c2     0|9|2  ->  [0 9 2]


## CSV with allele-specific copy number

If you have major/minor copy numbers per parental homolog — from CHISEL, for example —
use four fields instead of three:

```text
ref|alt|cn_maj|cn_min
```

The parsed array then has last dimension 4, and a missing allele-specific copy number is
written `.|.`. The two layouts must not be mixed within one file; `read_csv` raises if
they are. [](allele_specific.ipynb) covers what this buys you.

## VCF

`read_vcf` pulls per-cell counts out of a VCF's `FORMAT`/sample columns. By default it
reads the `AD` field, which it expects to hold three comma-separated values,
`ref,alt,cn`:

```text
GT:AD:DP    0/1:20,15,2:35
```

Sites whose `FORMAT` lacks the requested key are skipped with a message. Site names come
out as `CHROM:POS`, and cell names from the `#CHROM` header line.

In [6]:
vcf_reads, vcf_cells, vcf_sites = scna.util.read_vcf('data/test.vcf')

print("shape:", vcf_reads.shape)
print("cells:", vcf_cells)
print("sites:", vcf_sites)
print()
print(vcf_reads[0])

Info: Skipping line because AD not found in FORMAT: CHR[chr1], POS[250]
shape: (3, 5, 3)
cells: ['CELL_1', 'CELL_2', 'CELL_3', 'CELL_4', 'CELL_5']
sites: ['chr1:100', 'chr1:150', 'chr1:200']

[[20 15  2]
 [30  0  2]
 [ 0 25  2]
 [18 12  2]
 [40  0  2]]


Use `key=` to read a different `FORMAT` field:

```python
reads, cells, sites = scna.util.read_vcf('data/test.vcf', key='AD')
```